<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/23_GES_Aware_Genomic_RAG_Cell_7C16_Prespecified_Experiment2_RAG_Performance_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact frozen lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import math
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '23_GES_Aware_Genomic_RAG_Cell_7C16_'
    'Prespecified_Experiment2_RAG_Performance_Analysis.ipynb'
)
CELL_ID = '7C16'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_CONDITIONS = 6
EXPECTED_RUNS = 3
EXPECTED_QUESTION_CONDITION_UNITS = 480
BOOTSTRAP_REPLICATES = 2_000
BOOTSTRAP_SEED = 3035261668

PRIMARY_ENDPOINT = 'automated_evidence_fidelity_pass'

EXPECTED_CELL_7C15_TERMINAL_DECISION = (
    'PASS_STAGE7C15_CELL7C14_UNBLINDED_RESPONSE_TABLE_REVERIFIED_1440_ROWS_480_'
    'QUESTION_CONDITION_UNITS_THREE_RUNS_EACH_A004_ENDPOINT_AND_AGGREGATION_SPECS_'
    'REVERIFIED_80_FROZEN_QUESTION_GENE_METADATA_ALIGNED_CELL7C16_RAG_PERFORMANCE_'
    'ANALYSIS_AUTHORIZED_D_MINUS_A_PRIMARY_D_MINUS_B_C_E_F_SECONDARY_D_MINUS_A_'
    'FROZEN_SECONDARY_METRICS_2000_PAIRED_QUESTION_BOOTSTRAP_DETERMINISTIC_SEED_'
    'GENE_DESCRIPTIVE_ONLY_EGFR_EXPLORATORY_NO_NEW_ENDPOINTS_ARM_COMPARISONS_OR_'
    'SUBGROUP_HYPOTHESIS_TESTS'
)

# --------------------------------------------------------------------------------------
# Exact successful Cell 7C15 package.
# --------------------------------------------------------------------------------------
CELL_7C15_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c15_aggregation_performance_bootstrap_authorization_v1'
)
CELL_7C15_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c15_aggregation_performance_bootstrap_authorization_v1'
)

CELL_7C15 = OrderedDict([
    ('authorization', {
        'path': CELL_7C15_CONFIG_DIR / 'cell_7c15_aggregation_performance_bootstrap_authorization_v1.json',
        'sha256': 'a3b1550d23fabaf1e5d52c5dc3ec465f13968181a6ad0ba20af56de0308d618c',
    }),
    ('analysis_plan_snapshot', {
        'path': CELL_7C15_CONFIG_DIR / 'cell_7c15_authorized_analysis_plan_snapshot_v1.json',
        'sha256': 'e6a8e63f363c7c0919ac40d670c3c17b09c98df7c370bc520b716a783b09e88c',
    }),
    ('verified_input_inventory', {
        'path': CELL_7C15_CONFIG_DIR / 'cell_7c15_verified_input_inventory_v1.csv',
        'sha256': '3aa281dafba304238380c0ebc50a719f5e48d3f2ffbabc8042b39af6be5f8e1e',
    }),
    ('qc', {
        'path': CELL_7C15_QC_DIR / 'cell_7c15_aggregation_performance_bootstrap_authorization_qc_v1.json',
        'sha256': '3013210613a17b30112ba0911d4d91420dafc207366b302a130438c5c3dd6119',
    }),
    ('manifest', {
        'path': CELL_7C15_CONFIG_DIR / 'cell_7c15_aggregation_performance_bootstrap_authorization_manifest_v1.json',
        'sha256': '5b15a297214465b8575b894d05fb9a4a5e4cd8599d417460d354794c1f81c079',
    }),
])

# --------------------------------------------------------------------------------------
# Exact Cell 7C14 unblinded response-level input.
# --------------------------------------------------------------------------------------
CELL_7C14_TABLE = {
    'path': (
        ROOT / 'data_processed' / 'stage7_rag'
        / 'cell_7c14_a004_unblinded_response_level_table_v1'
        / 'cell_7c14_a004_unblinded_response_level_outcomes_v1.parquet'
    ),
    'sha256': '101bd5b16730a18b490894ed9a91d5caaface93e4c5c3127b8188e3022d9eee1',
}

# --------------------------------------------------------------------------------------
# Cell 7C16 outputs.
# --------------------------------------------------------------------------------------
DATA_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c16_a004_prespecified_rag_performance_analysis_v1'
)
RESULTS_DIR = (
    ROOT / 'outputs' / 'results' / 'stage7_rag'
    / 'cell_7c16_a004_prespecified_rag_performance_analysis_v1'
)
CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c16_a004_prespecified_rag_performance_analysis_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c16_a004_prespecified_rag_performance_analysis_v1'
)

OUTPUTS = OrderedDict([
    ('question_condition_metrics',
     DATA_DIR / 'cell_7c16_question_condition_metrics_v1.parquet'),
    ('condition_metric_summary',
     RESULTS_DIR / 'cell_7c16_condition_metric_summary_v1.csv'),
    ('comparison_results',
     RESULTS_DIR / 'cell_7c16_primary_and_secondary_comparison_results_v1.csv'),
    ('bootstrap_draw_audit',
     DATA_DIR / 'cell_7c16_bootstrap_question_draw_audit_v1.parquet'),
    ('bootstrap_replicate_estimates',
     DATA_DIR / 'cell_7c16_bootstrap_replicate_estimates_v1.parquet'),
    ('gene_descriptive_summary',
     RESULTS_DIR / 'cell_7c16_gene_stratified_descriptive_summary_v1.csv'),
    ('response_policy_distribution',
     RESULTS_DIR / 'cell_7c16_response_policy_distribution_v1.csv'),
    ('evidence_strength_distribution',
     RESULTS_DIR / 'cell_7c16_evidence_strength_distribution_v1.csv'),
    ('input_inventory',
     CONFIG_DIR / 'cell_7c16_verified_input_inventory_v1.csv'),
    ('execution_report',
     QC_DIR / 'cell_7c16_prespecified_rag_performance_execution_report_v1.json'),
    ('qc',
     QC_DIR / 'cell_7c16_prespecified_rag_performance_qc_v1.json'),
    ('manifest',
     CONFIG_DIR / 'cell_7c16_prespecified_rag_performance_manifest_v1.json'),
])

for directory in (DATA_DIR, RESULTS_DIR, CONFIG_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C16 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Data directory   : {DATA_DIR}')
print(f'Results directory: {RESULTS_DIR}')
print(f'Config directory : {CONFIG_DIR}')
print(f'QC directory     : {QC_DIR}')

Data directory   : /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage7_rag/cell_7c16_a004_prespecified_rag_performance_analysis_v1
Results directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/results/stage7_rag/cell_7c16_a004_prespecified_rag_performance_analysis_v1
Config directory : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c16_a004_prespecified_rag_performance_analysis_v1
QC directory     : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c16_a004_prespecified_rag_performance_analysis_v1


## 2. SHA-256, stable serialization, and statistical helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(
    label: str,
    path: Path,
    expected_sha256: str,
    require_sidecar: bool = True,
) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if require_sidecar and not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)) if sidecar_path(path).exists() else '',
        'sidecar_valid': sidecar_is_valid(path) if sidecar_path(path).exists() else False,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    frame.to_parquet(path, index=False, engine='pyarrow', compression='zstd')
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


def canonical_json_sha256(value: Any) -> str:
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=False,
        separators=(',', ':'),
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def percentile_ci(values: np.ndarray) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    if arr.shape != (BOOTSTRAP_REPLICATES,):
        raise AssertionError(
            f'Expected {BOOTSTRAP_REPLICATES} bootstrap estimates; observed {arr.shape}.'
        )
    lower, upper = np.quantile(arr, [0.025, 0.975], method='linear')
    return float(lower), float(upper)


with tempfile.TemporaryDirectory(prefix='cell_7c16_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / statistical helper self-test: PASS')

Serialization / statistical helper self-test: PASS


## 3. Reverify Cell 7C15 authorization and recover exact frozen question metadata

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C15.items():
    record = verify_exact_artifact(
        f'cell_7c15_{artifact_id}',
        spec['path'],
        spec['sha256'],
        require_sidecar=True,
    )
    record['source_cell'] = '7C15'
    verified_inputs.append(record)

authorization_7c15 = load_json(CELL_7C15['authorization']['path'])
analysis_plan = load_json(CELL_7C15['analysis_plan_snapshot']['path'])
manifest_7c15 = load_json(CELL_7C15['manifest']['path'])
qc_7c15 = load_json(CELL_7C15['qc']['path'])

if manifest_7c15.get('terminal_decision') != EXPECTED_CELL_7C15_TERMINAL_DECISION:
    raise AssertionError('Cell 7C15 terminal PASS mismatch.')
if manifest_7c15.get('next_authorized_cell') != '7C16':
    raise AssertionError('Cell 7C15 does not authorize Cell 7C16.')
if manifest_7c15.get('run_aggregation_authorized_in_7c16') is not True:
    raise AssertionError('Cell 7C15 does not authorize run aggregation.')
if manifest_7c15.get('primary_D_vs_A_authorized_in_7c16') is not True:
    raise AssertionError('Cell 7C15 does not authorize primary D-vs-A.')
if manifest_7c15.get('mandatory_secondary_D_vs_B_C_E_F_authorized_in_7c16') is not True:
    raise AssertionError('Cell 7C15 does not authorize mandatory secondary arm comparisons.')
if manifest_7c15.get('paired_bootstrap_2000_authorized_in_7c16') is not True:
    raise AssertionError('Cell 7C15 does not authorize paired bootstrap.')
if manifest_7c15.get('gene_specific_inferential_testing_authorized_in_7c16') is not False:
    raise AssertionError('Cell 7C15 unexpectedly authorizes gene-specific inference.')
if int(qc_7c15.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C15 QC does not report zero failures.')

if analysis_plan['bootstrap']['replicates'] != BOOTSTRAP_REPLICATES:
    raise AssertionError('Bootstrap replicate count mismatch.')
if analysis_plan['bootstrap']['seed'] != BOOTSTRAP_SEED:
    raise AssertionError('Bootstrap seed mismatch.')
if analysis_plan['primary_endpoint'] != PRIMARY_ENDPOINT:
    raise AssertionError('Primary endpoint mismatch.')

# Exact question artifact path was already verified and frozen by Cell 7C15.
question_path = Path(
    authorization_7c15['authorization_basis']['primary_question_set_path']
)
question_sha = authorization_7c15['authorization_basis']['primary_question_set_sha256']

question_record = verify_exact_artifact(
    'cell_7b3_primary_question_set',
    question_path,
    question_sha,
    require_sidecar=False,
)
question_record['source_cell'] = '7B3'
verified_inputs.append(question_record)

print('Cell 7C15 authorization package       : 5/5 exact hashes + sidecars')
print('Cell 7C15 terminal PASS               : VERIFIED')
print('Cell 7C16 performance authorization   : VERIFIED')
print(f'Frozen question metadata              : {question_path}')
print(f'Bootstrap seed                        : {BOOTSTRAP_SEED}')

Cell 7C15 authorization package       : 5/5 exact hashes + sidecars
Cell 7C15 terminal PASS               : VERIFIED
Cell 7C16 performance authorization   : VERIFIED
Frozen question metadata              : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage7_rag/cell_7b3_primary_question_set_v1.csv
Bootstrap seed                        : 3035261668


## 4. Reverify and load the frozen Cell 7C14 response-level table

In [5]:
record = verify_exact_artifact(
    'cell_7c14_unblinded_response_level_table',
    CELL_7C14_TABLE['path'],
    CELL_7C14_TABLE['sha256'],
    require_sidecar=True,
)
record['source_cell'] = '7C14'
verified_inputs.append(record)

responses = pd.read_parquet(CELL_7C14_TABLE['path'])
questions = pd.read_csv(question_path, dtype=str).fillna('')

REQUIRED_RESPONSE_COLUMNS = {
    'review_item_id',
    'question_id',
    'run_id',
    'blinded_alias',
    'condition_id',
    'condition_name',
    PRIMARY_ENDPOINT,
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
    'context_citation_coverage',
    'invalid_or_hallucinated_evidence_id_count',
    'confidence',
    'response_policy',
    'evidence_strength',
}
missing_response = sorted(REQUIRED_RESPONSE_COLUMNS - set(responses.columns))
if missing_response:
    raise AssertionError(
        'Cell 7C14 response table missing required columns: '
        + ', '.join(missing_response)
    )

REQUIRED_QUESTION_COLUMNS = {'question_id', 'target_gene'}
missing_question = sorted(REQUIRED_QUESTION_COLUMNS - set(questions.columns))
if missing_question:
    raise AssertionError(
        'Frozen question metadata missing required columns: '
        + ', '.join(missing_question)
    )

if len(responses) != EXPECTED_RESPONSES:
    raise AssertionError('Expected 1,440 response rows.')
if responses['review_item_id'].nunique() != EXPECTED_RESPONSES:
    raise AssertionError('review_item_id is not unique.')
if responses['question_id'].nunique() != EXPECTED_QUESTIONS:
    raise AssertionError('Expected 80 question IDs.')
if len(questions) != EXPECTED_QUESTIONS or questions['question_id'].nunique() != EXPECTED_QUESTIONS:
    raise AssertionError('Frozen question table must contain exactly 80 unique question IDs.')

responses['run_id'] = pd.to_numeric(responses['run_id'], errors='raise').astype(int)
if set(responses['run_id'].unique()) != {0, 1, 2}:
    raise AssertionError('Run IDs must be exactly 0,1,2.')
if set(responses['condition_id'].astype(str)) != {'A', 'B', 'C', 'D', 'E', 'F'}:
    raise AssertionError('Condition IDs must be exactly A-F.')

qcond_counts = responses.groupby(['question_id', 'condition_id']).size()
if len(qcond_counts) != EXPECTED_QUESTION_CONDITION_UNITS:
    raise AssertionError('Expected 480 question-condition groups.')
if not qcond_counts.eq(EXPECTED_RUNS).all():
    raise AssertionError('Every question-condition group must contain exactly 3 runs.')

question_ids_response = set(responses['question_id'].astype(str))
question_ids_meta = set(questions['question_id'].astype(str))
if question_ids_response != question_ids_meta:
    raise AssertionError('Response/question metadata question IDs do not align exactly.')

print('Cell 7C14 response table              : exact SHA + sidecar verified')
print('Response observations                 : 1,440')
print('Question-condition groups             : 480')
print('Runs per question-condition           : 3')
print('Question metadata alignment            : EXACT')

Cell 7C14 response table              : exact SHA + sidecar verified
Response observations                 : 1,440
Question-condition groups             : 480
Runs per question-condition           : 3
Question metadata alignment            : EXACT


## 5. Aggregate the three runs into the frozen 480 question-condition analysis units

In [6]:
INFERENTIAL_AND_DESCRIPTIVE_METRICS = [
    PRIMARY_ENDPOINT,
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
    'context_citation_coverage',
    'invalid_or_hallucinated_evidence_id_count',
    'confidence',
]

BINARY_METRICS = [
    PRIMARY_ENDPOINT,
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
]

# Convert only the frozen authorized metrics to numeric values.
working = responses.copy()
for metric in BINARY_METRICS:
    working[metric] = working[metric].astype(bool).astype(float)

for metric in [
    'context_citation_coverage',
    'invalid_or_hallucinated_evidence_id_count',
    'confidence',
]:
    working[metric] = pd.to_numeric(working[metric], errors='raise').astype(float)

group_keys = [
    'question_id',
    'condition_id',
    'condition_name',
    'blinded_alias',
]

question_condition = (
    working
    .groupby(group_keys, as_index=False, sort=True)
    .agg(
        run_count=('run_id', 'count'),
        **{
            metric: (metric, 'mean')
            for metric in INFERENTIAL_AND_DESCRIPTIVE_METRICS
        },
    )
)

if len(question_condition) != EXPECTED_QUESTION_CONDITION_UNITS:
    raise AssertionError(
        f'Expected 480 question-condition rows; observed {len(question_condition)}.'
    )
if not question_condition['run_count'].eq(3).all():
    raise AssertionError('Every aggregated question-condition row must have run_count=3.')

gene_map = (
    questions[['question_id', 'target_gene']]
    .copy()
    .drop_duplicates('question_id')
)
question_condition = question_condition.merge(
    gene_map,
    on='question_id',
    how='left',
    validate='many_to_one',
)
if question_condition['target_gene'].eq('').any() or question_condition['target_gene'].isna().any():
    raise AssertionError('Missing target_gene after question metadata join.')

# Ensure exactly one row for every question x condition.
if (
    question_condition.groupby(['question_id', 'condition_id']).size().ne(1).any()
):
    raise AssertionError('Question-condition table is not one row per question x condition.')

print('Question-condition analysis units      : 480')
print('Run aggregation rule                   : arithmetic mean of runs 0,1,2')
print('Primary endpoint aggregated            : YES')
print('Six frozen secondary metrics aggregated: YES')
print('Confidence aggregated                  : descriptive only')

Question-condition analysis units      : 480
Run aggregation rule                   : arithmetic mean of runs 0,1,2
Primary endpoint aggregated            : YES
Six frozen secondary metrics aggregated: YES
Confidence aggregated                  : descriptive only


## 6. Freeze condition-level descriptive summaries

In [7]:
summary_rows = []

for condition_id, condition_frame in question_condition.groupby('condition_id', sort=True):
    condition_name = str(condition_frame['condition_name'].iloc[0])

    if len(condition_frame) != EXPECTED_QUESTIONS:
        raise AssertionError(
            f'Condition {condition_id} has {len(condition_frame)} question rows; expected 80.'
        )

    for metric in INFERENTIAL_AND_DESCRIPTIVE_METRICS:
        values = condition_frame[metric].to_numpy(dtype=float)

        summary_rows.append({
            'condition_id': str(condition_id),
            'condition_name': condition_name,
            'metric': metric,
            'n_questions': int(len(values)),
            'mean': float(np.mean(values)),
            'sd_across_questions': float(np.std(values, ddof=1)),
            'min': float(np.min(values)),
            'max': float(np.max(values)),
            'inferential_status': (
                'primary'
                if metric == PRIMARY_ENDPOINT
                else 'authorized_D_vs_A_secondary'
                if metric in analysis_plan['secondary_D_vs_A_inferential_metrics']
                and metric != 'confidence'
                else 'descriptive_only'
            ),
        })

condition_summary = pd.DataFrame(summary_rows)

expected_summary_rows = EXPECTED_CONDITIONS * len(INFERENTIAL_AND_DESCRIPTIVE_METRICS)
if len(condition_summary) != expected_summary_rows:
    raise AssertionError(
        f'Expected {expected_summary_rows} condition-summary rows; observed {len(condition_summary)}.'
    )

# Response-policy descriptive distribution.
response_policy_distribution = (
    responses
    .groupby(['condition_id', 'condition_name', 'response_policy'], as_index=False)
    .size()
    .rename(columns={'size': 'n_responses'})
)
response_policy_distribution['condition_total'] = (
    response_policy_distribution.groupby('condition_id')['n_responses'].transform('sum')
)
response_policy_distribution['proportion'] = (
    response_policy_distribution['n_responses']
    / response_policy_distribution['condition_total']
)

# Evidence-strength descriptive distribution.
evidence_strength_distribution = (
    responses
    .groupby(['condition_id', 'condition_name', 'evidence_strength'], as_index=False)
    .size()
    .rename(columns={'size': 'n_responses'})
)
evidence_strength_distribution['condition_total'] = (
    evidence_strength_distribution.groupby('condition_id')['n_responses'].transform('sum')
)
evidence_strength_distribution['proportion'] = (
    evidence_strength_distribution['n_responses']
    / evidence_strength_distribution['condition_total']
)

if not response_policy_distribution.groupby('condition_id')['n_responses'].sum().eq(240).all():
    raise AssertionError('Each condition must have 240 response-policy observations.')
if not evidence_strength_distribution.groupby('condition_id')['n_responses'].sum().eq(240).all():
    raise AssertionError('Each condition must have 240 evidence-strength observations.')

print('Condition-level metric summaries       : MATERIALIZED')
print('Response-policy distributions           : MATERIALIZED')
print('Evidence-strength distributions         : MATERIALIZED')

Condition-level metric summaries       : MATERIALIZED
Response-policy distributions           : MATERIALIZED
Evidence-strength distributions         : MATERIALIZED


## 7. Create one deterministic paired bootstrap draw schedule shared by every comparison

In [8]:
question_ids = sorted(question_condition['question_id'].astype(str).unique())
if len(question_ids) != EXPECTED_QUESTIONS:
    raise AssertionError('Expected exactly 80 ordered question IDs for bootstrap.')

rng = np.random.default_rng(BOOTSTRAP_SEED)
draw_indices = rng.integers(
    low=0,
    high=EXPECTED_QUESTIONS,
    size=(BOOTSTRAP_REPLICATES, EXPECTED_QUESTIONS),
    endpoint=False,
)

if draw_indices.shape != (2000, 80):
    raise AssertionError(f'Unexpected bootstrap draw matrix shape: {draw_indices.shape}')

bootstrap_draw_rows = []
for replicate_idx in range(BOOTSTRAP_REPLICATES):
    sampled_ids = [question_ids[i] for i in draw_indices[replicate_idx]]
    sampled_json = json.dumps(
        sampled_ids,
        ensure_ascii=False,
        separators=(',', ':'),
    )
    bootstrap_draw_rows.append({
        'bootstrap_replicate': replicate_idx + 1,
        'sampled_question_count': EXPECTED_QUESTIONS,
        'sampled_question_ids_json': sampled_json,
        'sampled_question_ids_sha256': hashlib.sha256(
            sampled_json.encode('utf-8')
        ).hexdigest(),
    })

bootstrap_draw_audit = pd.DataFrame(bootstrap_draw_rows)

if len(bootstrap_draw_audit) != BOOTSTRAP_REPLICATES:
    raise AssertionError('Bootstrap draw audit must contain exactly 2,000 rows.')
if bootstrap_draw_audit['sampled_question_ids_sha256'].nunique() < 1900:
    raise AssertionError(
        'Unexpectedly low uniqueness among 2,000 bootstrap question samples.'
    )

print('Bootstrap replicates                  : 2,000')
print('Questions sampled per replicate        : 80 with replacement')
print('Paired question draws                  : YES')
print('Same draw schedule for all comparisons : YES')
print(f'Frozen deterministic seed              : {BOOTSTRAP_SEED}')

Bootstrap replicates                  : 2,000
Questions sampled per replicate        : 80 with replacement
Paired question draws                  : YES
Same draw schedule for all comparisons : YES
Frozen deterministic seed              : 3035261668


## 8. Calculate the primary, mandatory secondary-arm, and frozen secondary-metric comparisons

In [9]:
CONDITION_NAMES = {
    'A': 'semantic-only',
    'B': 'review/conflict-aware',
    'C': 'combined-metadata',
    'D': 'Full-GES',
    'E': 'no-star-GES',
    'F': 'random-quality',
}

AUTHORIZED_COMPARISONS = [
    {
        'comparison_id': 'PRIMARY_D_MINUS_A',
        'comparison_family': 'primary',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'A',
        'favorable_direction': 'higher',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_B',
        'comparison_family': 'mandatory_secondary_arm',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'B',
        'favorable_direction': 'higher',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_C',
        'comparison_family': 'mandatory_secondary_arm',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'C',
        'favorable_direction': 'higher',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_E',
        'comparison_family': 'mandatory_secondary_arm',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'E',
        'favorable_direction': 'higher',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_F',
        'comparison_family': 'mandatory_secondary_arm',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'F',
        'favorable_direction': 'higher',
    },
]

for metric in analysis_plan['secondary_D_vs_A_inferential_metrics']:
    if metric == 'confidence':
        continue
    AUTHORIZED_COMPARISONS.append({
        'comparison_id': f'SECONDARY_METRIC_D_MINUS_A__{metric}',
        'comparison_family': 'secondary_metric_D_vs_A',
        'metric': metric,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'A',
        'favorable_direction': (
            'lower'
            if metric == 'invalid_or_hallucinated_evidence_id_count'
            else 'higher'
        ),
    })

if len(AUTHORIZED_COMPARISONS) != 11:
    raise AssertionError(
        f'Expected 11 authorized comparisons; observed {len(AUTHORIZED_COMPARISONS)}.'
    )

# Build exact question x condition lookup for fast paired calculations.
metric_lookup: dict[tuple[str, str], dict[str, np.ndarray]] = {}

for condition_id in ['A', 'B', 'C', 'D', 'E', 'F']:
    frame = (
        question_condition.loc[
            question_condition['condition_id'].eq(condition_id),
            ['question_id'] + INFERENTIAL_AND_DESCRIPTIVE_METRICS,
        ]
        .copy()
        .set_index('question_id')
        .reindex(question_ids)
    )

    if frame.isna().any().any():
        raise AssertionError(f'Missing question-level metric for condition {condition_id}.')

    for metric in INFERENTIAL_AND_DESCRIPTIVE_METRICS:
        metric_lookup[(condition_id, metric)] = frame[metric].to_numpy(dtype=float)

comparison_rows = []
bootstrap_estimate_frames = []

for spec in AUTHORIZED_COMPARISONS:
    metric = spec['metric']
    exp_id = spec['experimental_condition_id']
    ref_id = spec['reference_condition_id']

    experimental = metric_lookup[(exp_id, metric)]
    reference = metric_lookup[(ref_id, metric)]
    paired_difference = experimental - reference

    point_estimate = float(np.mean(paired_difference))

    # Paired bootstrap: the same sampled question indices are applied to both arms.
    bootstrap_estimates = paired_difference[draw_indices].mean(axis=1)
    ci_lower, ci_upper = percentile_ci(bootstrap_estimates)

    experimental_mean = float(np.mean(experimental))
    reference_mean = float(np.mean(reference))

    if spec['favorable_direction'] == 'higher':
        effect_favors_experimental = point_estimate > 0
    else:
        effect_favors_experimental = point_estimate < 0

    comparison_rows.append({
        'comparison_id': spec['comparison_id'],
        'comparison_family': spec['comparison_family'],
        'metric': metric,
        'experimental_condition_id': exp_id,
        'experimental_condition_name': CONDITION_NAMES[exp_id],
        'reference_condition_id': ref_id,
        'reference_condition_name': CONDITION_NAMES[ref_id],
        'n_paired_questions': EXPECTED_QUESTIONS,
        'experimental_mean': experimental_mean,
        'reference_mean': reference_mean,
        'paired_difference_experimental_minus_reference': point_estimate,
        'bootstrap_ci_2_5': ci_lower,
        'bootstrap_ci_97_5': ci_upper,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'favorable_direction': spec['favorable_direction'],
        'point_estimate_favors_experimental': bool(effect_favors_experimental),
        'ci_excludes_zero': bool(ci_lower > 0 or ci_upper < 0),
        'p_value_calculated': False,
    })

    bootstrap_estimate_frames.append(
        pd.DataFrame({
            'bootstrap_replicate': np.arange(1, BOOTSTRAP_REPLICATES + 1, dtype=int),
            'comparison_id': spec['comparison_id'],
            'comparison_family': spec['comparison_family'],
            'metric': metric,
            'experimental_condition_id': exp_id,
            'reference_condition_id': ref_id,
            'paired_difference_experimental_minus_reference': bootstrap_estimates.astype(float),
        })
    )

comparison_results = pd.DataFrame(comparison_rows)
bootstrap_replicate_estimates = pd.concat(
    bootstrap_estimate_frames,
    ignore_index=True,
)

if len(comparison_results) != 11:
    raise AssertionError('Comparison results must contain exactly 11 rows.')
if len(bootstrap_replicate_estimates) != 11 * BOOTSTRAP_REPLICATES:
    raise AssertionError('Bootstrap estimate table must contain exactly 22,000 rows.')
if comparison_results['p_value_calculated'].any():
    raise AssertionError('No p-values are authorized.')

print('Primary D-vs-A comparison              : CALCULATED')
print('Mandatory D-vs-B/C/E/F comparisons     : CALCULATED')
print('Frozen secondary-metric D-vs-A         : 6 CALCULATED')
print('Bootstrap replicate estimates           : 22,000')
print('P-values                                : NOT CALCULATED')

Primary D-vs-A comparison              : CALCULATED
Mandatory D-vs-B/C/E/F comparisons     : CALCULATED
Frozen secondary-metric D-vs-A         : 6 CALCULATED
Bootstrap replicate estimates           : 22,000
P-values                                : NOT CALCULATED


## 9. Gene-stratified descriptive summaries — no subgroup inference

In [10]:
GENE_REPORTING_METRICS = [
    PRIMARY_ENDPOINT,
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
    'context_citation_coverage',
    'invalid_or_hallucinated_evidence_id_count',
]

expected_gene_set = {'BRCA1', 'BRCA2', 'MLH1', 'EGFR'}
observed_gene_set = set(question_condition['target_gene'].astype(str))
if observed_gene_set != expected_gene_set:
    raise AssertionError(
        f'Unexpected gene set: {sorted(observed_gene_set)}'
    )

gene_rows = []
for (gene, condition_id), frame in question_condition.groupby(
    ['target_gene', 'condition_id'],
    sort=True,
):
    condition_name = str(frame['condition_name'].iloc[0])
    reporting_status = 'exploratory' if gene == 'EGFR' else 'prespecified_descriptive'

    for metric in GENE_REPORTING_METRICS:
        values = frame[metric].to_numpy(dtype=float)
        gene_rows.append({
            'target_gene': str(gene),
            'gene_reporting_status': reporting_status,
            'condition_id': str(condition_id),
            'condition_name': condition_name,
            'metric': metric,
            'n_questions': int(len(values)),
            'mean': float(np.mean(values)),
            'sd_across_questions': (
                float(np.std(values, ddof=1))
                if len(values) > 1
                else float('nan')
            ),
            'inferential_test_performed': False,
            'bootstrap_ci_calculated': False,
        })

gene_summary = pd.DataFrame(gene_rows)

expected_gene_rows = 4 * 6 * len(GENE_REPORTING_METRICS)
if len(gene_summary) != expected_gene_rows:
    raise AssertionError(
        f'Expected {expected_gene_rows} gene descriptive rows; observed {len(gene_summary)}.'
    )
if gene_summary['inferential_test_performed'].any():
    raise AssertionError('Gene-specific inferential testing is prohibited.')
if gene_summary['bootstrap_ci_calculated'].any():
    raise AssertionError('Gene-specific bootstrap CIs are not authorized.')

print('Gene-stratified descriptive rows       :', len(gene_summary))
print('BRCA1 / BRCA2 / MLH1                  : prespecified descriptive')
print('EGFR                                   : exploratory')
print('Gene-specific inference                : NOT PERFORMED')

Gene-stratified descriptive rows       : 168
BRCA1 / BRCA2 / MLH1                  : prespecified descriptive
EGFR                                   : exploratory
Gene-specific inference                : NOT PERFORMED


## 10. Pre-freeze scientific and structural QC

In [11]:
authorized_comparison_ids = {
    'PRIMARY_D_MINUS_A',
    'SECONDARY_D_MINUS_B',
    'SECONDARY_D_MINUS_C',
    'SECONDARY_D_MINUS_E',
    'SECONDARY_D_MINUS_F',
    'SECONDARY_METRIC_D_MINUS_A__citation_integrity_pass',
    'SECONDARY_METRIC_D_MINUS_A__citation_presence_pass',
    'SECONDARY_METRIC_D_MINUS_A__conflict_concordance_pass',
    'SECONDARY_METRIC_D_MINUS_A__caution_policy_concordance_pass',
    'SECONDARY_METRIC_D_MINUS_A__context_citation_coverage',
    'SECONDARY_METRIC_D_MINUS_A__invalid_or_hallucinated_evidence_id_count',
}

qc_checks = OrderedDict([
    ('response_rows_1440', len(responses) == 1440),
    ('questions_80', responses['question_id'].nunique() == 80),
    ('conditions_6', responses['condition_id'].nunique() == 6),
    ('question_condition_rows_480', len(question_condition) == 480),
    ('three_runs_per_question_condition', question_condition['run_count'].eq(3).all()),
    ('condition_summary_expected_rows',
     len(condition_summary) == 6 * len(INFERENTIAL_AND_DESCRIPTIVE_METRICS)),
    ('primary_comparison_present_once',
     comparison_results['comparison_id'].eq('PRIMARY_D_MINUS_A').sum() == 1),
    ('comparison_count_11', len(comparison_results) == 11),
    ('comparison_ids_exact',
     set(comparison_results['comparison_id']) == authorized_comparison_ids),
    ('primary_endpoint_comparison_count_5',
     comparison_results['metric'].eq(PRIMARY_ENDPOINT).sum() == 5),
    ('secondary_DA_metric_count_6',
     comparison_results['comparison_family'].eq('secondary_metric_D_vs_A').sum() == 6),
    ('all_comparisons_D_experimental',
     comparison_results['experimental_condition_id'].eq('D').all()),
    ('bootstrap_draw_rows_2000', len(bootstrap_draw_audit) == 2000),
    ('bootstrap_estimate_rows_22000', len(bootstrap_replicate_estimates) == 22000),
    ('bootstrap_2000_each_comparison',
     bootstrap_replicate_estimates.groupby('comparison_id').size().eq(2000).all()),
    ('bootstrap_replicate_range_exact',
     set(bootstrap_replicate_estimates['bootstrap_replicate'].unique())
     == set(range(1, 2001))),
    ('bootstrap_seed_exact', BOOTSTRAP_SEED == 3035261668),
    ('comparison_ci_complete',
     comparison_results[['bootstrap_ci_2_5', 'bootstrap_ci_97_5']].notna().all().all()),
    ('comparison_ci_ordered',
     (comparison_results['bootstrap_ci_2_5'] <= comparison_results['bootstrap_ci_97_5']).all()),
    ('p_values_not_calculated',
     comparison_results['p_value_calculated'].eq(False).all()),
    ('gene_rows_expected', len(gene_summary) == 4 * 6 * len(GENE_REPORTING_METRICS)),
    ('gene_set_exact', set(gene_summary['target_gene']) == {'BRCA1', 'BRCA2', 'MLH1', 'EGFR'}),
    ('egfr_exploratory_only',
     gene_summary.loc[gene_summary['target_gene'].eq('EGFR'), 'gene_reporting_status']
     .eq('exploratory').all()),
    ('gene_inference_false', gene_summary['inferential_test_performed'].eq(False).all()),
    ('gene_bootstrap_false', gene_summary['bootstrap_ci_calculated'].eq(False).all()),
    ('response_policy_240_per_condition',
     response_policy_distribution.groupby('condition_id')['n_responses'].sum().eq(240).all()),
    ('evidence_strength_240_per_condition',
     evidence_strength_distribution.groupby('condition_id')['n_responses'].sum().eq(240).all()),
    ('no_new_endpoint_family', True),
    ('no_new_arm_comparison_family', True),
    ('no_human_review', True),
    ('no_llm_judge', True),
])

failed = [name for name, passed in qc_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C16 pre-freeze QC failed:\\n- ' + '\\n- '.join(failed)
    )

print(f'Pre-freeze QC checks                  : {len(qc_checks)}/{len(qc_checks)} PASS')

Pre-freeze QC checks                  : 31/31 PASS


## 11. Freeze the complete Experiment 2 RAG performance-analysis package

In [12]:
stable_write_parquet(
    OUTPUTS['question_condition_metrics'],
    question_condition,
)
write_sidecar(OUTPUTS['question_condition_metrics'])

stable_write_csv(
    OUTPUTS['condition_metric_summary'],
    condition_summary,
)
write_sidecar(OUTPUTS['condition_metric_summary'])

stable_write_csv(
    OUTPUTS['comparison_results'],
    comparison_results,
)
write_sidecar(OUTPUTS['comparison_results'])

stable_write_parquet(
    OUTPUTS['bootstrap_draw_audit'],
    bootstrap_draw_audit,
)
write_sidecar(OUTPUTS['bootstrap_draw_audit'])

stable_write_parquet(
    OUTPUTS['bootstrap_replicate_estimates'],
    bootstrap_replicate_estimates,
)
write_sidecar(OUTPUTS['bootstrap_replicate_estimates'])

stable_write_csv(
    OUTPUTS['gene_descriptive_summary'],
    gene_summary,
)
write_sidecar(OUTPUTS['gene_descriptive_summary'])

stable_write_csv(
    OUTPUTS['response_policy_distribution'],
    response_policy_distribution,
)
write_sidecar(OUTPUTS['response_policy_distribution'])

stable_write_csv(
    OUTPUTS['evidence_strength_distribution'],
    evidence_strength_distribution,
)
write_sidecar(OUTPUTS['evidence_strength_distribution'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(
    OUTPUTS['input_inventory'],
    input_inventory,
)
write_sidecar(OUTPUTS['input_inventory'])

primary_result = (
    comparison_results
    .loc[comparison_results['comparison_id'].eq('PRIMARY_D_MINUS_A')]
    .iloc[0]
    .to_dict()
)

execution_report = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'analysis_population': {
        'response_rows': 1440,
        'questions': 80,
        'conditions': 6,
        'runs_per_question_condition': 3,
        'question_condition_units': 480,
    },
    'primary_endpoint': PRIMARY_ENDPOINT,
    'primary_result': {
        'experimental_condition': 'D Full-GES',
        'reference_condition': 'A semantic-only',
        'experimental_mean': float(primary_result['experimental_mean']),
        'reference_mean': float(primary_result['reference_mean']),
        'paired_difference_D_minus_A':
            float(primary_result['paired_difference_experimental_minus_reference']),
        'bootstrap_ci_2_5': float(primary_result['bootstrap_ci_2_5']),
        'bootstrap_ci_97_5': float(primary_result['bootstrap_ci_97_5']),
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'ci_excludes_zero': bool(primary_result['ci_excludes_zero']),
    },
    'comparison_counts': {
        'primary': 1,
        'mandatory_secondary_arm': 4,
        'secondary_metric_D_vs_A': 6,
        'total': 11,
    },
    'bootstrap': {
        'seed': BOOTSTRAP_SEED,
        'replicates': BOOTSTRAP_REPLICATES,
        'paired_by_question': True,
        'shared_draw_schedule_across_comparisons': True,
        'bootstrap_replicate_estimate_rows': 22000,
    },
    'gene_reporting': {
        'BRCA1_BRCA2_MLH1': 'prespecified_descriptive',
        'EGFR': 'exploratory',
        'gene_specific_inferential_testing_performed': False,
    },
    'claim_boundary': {
        'free_text_factual_correctness_established': False,
        'semantic_citation_entailment_established': False,
        'clinical_appropriateness_established': False,
        'patient_level_safety_established': False,
    },
    'next_automated_execution_authorized': False,
}

stable_write_json(
    OUTPUTS['execution_report'],
    execution_report,
)
write_sidecar(OUTPUTS['execution_report'])

terminal_decision = (
    'PASS_STAGE7C16_EXPERIMENT2_A004_PRESPECIFIED_RAG_PERFORMANCE_ANALYSIS_COMPLETE_'
    '1440_RESPONSES_AGGREGATED_TO_480_QUESTION_CONDITION_UNITS_PRIMARY_D_MINUS_A_'
    'AUTOMATED_EVIDENCE_FIDELITY_AND_MANDATORY_D_MINUS_B_C_E_F_PLUS_SIX_FROZEN_'
    'SECONDARY_D_MINUS_A_METRICS_CALCULATED_WITH_2000_PAIRED_QUESTION_BOOTSTRAP_'
    'REPLICATES_GENE_DESCRIPTIVE_REPORTING_FROZEN_EGFR_EXPLORATORY_NO_GENE_'
    'INFERENCE_NEW_ENDPOINTS_NEW_ARM_COMPARISONS_HUMAN_REVIEW_LLM_JUDGE_FREE_TEXT_'
    'FACTUAL_CORRECTNESS_OR_SEMANTIC_CITATION_ENTAILMENT_CLAIMS_NEXT_AUTOMATED_'
    'EXECUTION_NOT_AUTHORIZED'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in qc_checks.items()},
    'passed_checks': len(qc_checks),
    'failed_checks': 0,
    'total_checks': len(qc_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c15_manifest_sha256': CELL_7C15['manifest']['sha256'],
        'cell_7c15_analysis_plan_snapshot_sha256':
            CELL_7C15['analysis_plan_snapshot']['sha256'],
        'cell_7c14_unblinded_response_level_table_sha256':
            CELL_7C14_TABLE['sha256'],
        'primary_question_set_sha256':
            authorization_7c15['authorization_basis']['primary_question_set_sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'experiment2_rag_performance_results_frozen': True,
    'primary_comparison_completed': True,
    'mandatory_secondary_arm_comparisons_completed': True,
    'secondary_D_vs_A_metric_comparisons_completed': True,
    'paired_bootstrap_2000_completed': True,
    'gene_specific_inferential_testing_performed': False,
    'new_endpoint_discovery_performed': False,
    'new_arm_comparisons_performed': False,
    'next_authorized_cell': None,
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh readback of every frozen output.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C16 final readback failed: {path}')

rb_qcond = pd.read_parquet(OUTPUTS['question_condition_metrics'])
rb_comp = pd.read_csv(OUTPUTS['comparison_results'])
rb_draws = pd.read_parquet(OUTPUTS['bootstrap_draw_audit'])
rb_boot = pd.read_parquet(OUTPUTS['bootstrap_replicate_estimates'])
rb_gene = pd.read_csv(OUTPUTS['gene_descriptive_summary'])
rb_report = load_json(OUTPUTS['execution_report'])
rb_qc = load_json(OUTPUTS['qc'])
rb_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('readback_qcond_480', len(rb_qcond) == 480),
    ('readback_comparisons_11', len(rb_comp) == 11),
    ('readback_draws_2000', len(rb_draws) == 2000),
    ('readback_bootstrap_22000', len(rb_boot) == 22000),
    ('readback_gene_rows', len(rb_gene) == expected_gene_rows),
    ('readback_primary_once',
     rb_comp['comparison_id'].eq('PRIMARY_D_MINUS_A').sum() == 1),
    ('readback_no_pvalues',
     rb_comp['p_value_calculated'].astype(str).str.lower().eq('false').all()),
    ('manifest_results_frozen',
     rb_manifest.get('experiment2_rag_performance_results_frozen') is True),
    ('manifest_next_none',
     rb_manifest.get('next_authorized_cell') is None),
    ('manifest_gene_inference_false',
     rb_manifest.get('gene_specific_inferential_testing_performed') is False),
    ('manifest_new_endpoints_false',
     rb_manifest.get('new_endpoint_discovery_performed') is False),
    ('manifest_new_arms_false',
     rb_manifest.get('new_arm_comparisons_performed') is False),
    ('qc_zero_failures',
     int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid',
     all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C16 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(qc_checks) + len(readback_checks)

# Human-readable result tables for the terminal output.
primary_display = rb_comp.loc[
    rb_comp['comparison_id'].eq('PRIMARY_D_MINUS_A')
].iloc[0]

arm_display = rb_comp.loc[
    rb_comp['metric'].eq(PRIMARY_ENDPOINT),
    [
        'comparison_id',
        'experimental_mean',
        'reference_mean',
        'paired_difference_experimental_minus_reference',
        'bootstrap_ci_2_5',
        'bootstrap_ci_97_5',
    ],
].copy()

secondary_metric_display = rb_comp.loc[
    rb_comp['comparison_family'].eq('secondary_metric_D_vs_A'),
    [
        'metric',
        'experimental_mean',
        'reference_mean',
        'paired_difference_experimental_minus_reference',
        'bootstrap_ci_2_5',
        'bootstrap_ci_97_5',
    ],
].copy()

condition_primary = (
    pd.read_csv(OUTPUTS['condition_metric_summary'])
    .loc[lambda d: d['metric'].eq(PRIMARY_ENDPOINT),
         ['condition_id', 'condition_name', 'mean', 'sd_across_questions']]
    .sort_values('condition_id')
)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C16')
print('PRESPECIFIED EXPERIMENT 2 RAG PERFORMANCE ANALYSIS — FINAL FROZEN RESULTS')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nANALYSIS POPULATION')
print('Frozen response observations                  : 1,440')
print('Primary questions                             : 80')
print('Experimental conditions                       : 6')
print('Runs per question-condition                   : 3')
print('Question-condition analysis units             : 480')

print('\\nPRIMARY ENDPOINT')
print('Endpoint                                      : automated_evidence_fidelity_pass')
print('Comparison                                    : D Full-GES minus A semantic-only')
print(f'D Full-GES mean                               : {float(primary_display["experimental_mean"]):.6f}')
print(f'A semantic-only mean                          : {float(primary_display["reference_mean"]):.6f}')
print(f'Paired difference D-A                         : {float(primary_display["paired_difference_experimental_minus_reference"]):.6f}')
print(
    '95% paired-question bootstrap CI             : '
    f'[{float(primary_display["bootstrap_ci_2_5"]):.6f}, '
    f'{float(primary_display["bootstrap_ci_97_5"]):.6f}]'
)
print(f'CI excludes zero                              : {bool(primary_display["ci_excludes_zero"])}')

print('\\nPRIMARY ENDPOINT — CONDITION DESCRIPTIVES')
for row in condition_primary.itertuples(index=False):
    print(
        f'{row.condition_id} {row.condition_name:<24} '
        f'mean={float(row.mean):.6f}  sd_q={float(row.sd_across_questions):.6f}'
    )

print('\\nPRIMARY-ENDPOINT ARM COMPARISONS')
for row in arm_display.itertuples(index=False):
    print(
        f'{row.comparison_id:<24} '
        f'diff={float(row.paired_difference_experimental_minus_reference): .6f}  '
        f'95% CI=[{float(row.bootstrap_ci_2_5): .6f}, {float(row.bootstrap_ci_97_5): .6f}]'
    )

print('\\nFROZEN SECONDARY-METRIC D-vs-A COMPARISONS')
for row in secondary_metric_display.itertuples(index=False):
    print(
        f'{row.metric:<46} '
        f'diff={float(row.paired_difference_experimental_minus_reference): .6f}  '
        f'95% CI=[{float(row.bootstrap_ci_2_5): .6f}, {float(row.bootstrap_ci_97_5): .6f}]'
    )

print('\\nBOOTSTRAP')
print('Replicates                                    : 2,000')
print('Resampling unit                               : question')
print('Paired                                        : YES')
print(f'Deterministic seed                            : {BOOTSTRAP_SEED}')
print('Shared draw schedule across comparisons       : YES')
print('P-values                                      : NOT CALCULATED')

print('\\nGENE REPORTING')
print('BRCA1 / BRCA2 / MLH1                          : prespecified descriptive')
print('EGFR                                          : exploratory / separate')
print('Gene-specific inferential tests               : NOT PERFORMED')

print('\\nCLAIM BOUNDARY')
print('Free-text factual correctness established     : NO')
print('Semantic citation entailment established      : NO')
print('Clinical appropriateness established          : NO')
print('Patient-level safety established              : NO')

print('\\nCELL 7C16 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT BOUNDARY')
print('Experiment 2 RAG performance results          : FROZEN')
print('Next automated cell                           : NOT AUTHORIZED')
print('Recommended next activity                     : manuscript/report tables, figures, and interpretation')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C16
PRESPECIFIED EXPERIMENT 2 RAG PERFORMANCE ANALYSIS — FINAL FROZEN RESULTS
Notebook                                      : 23_GES_Aware_Genomic_RAG_Cell_7C16_Prespecified_Experiment2_RAG_Performance_Analysis.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nANALYSIS POPULATION
Frozen response observations                  : 1,440
Primary questions                             : 80
Experimental conditions                       : 6
Runs per question-condition                   : 3
Question-condition analysis units             : 480
\nPRIMARY ENDPOINT
Endpoint                                      : automated_evidence_fidelity_pass
Comparison                                    : D Full-GES minus A semantic-only
D Full-GES mean                     